# 04 · Interaction extraction

Apply the collaborator-provided PLIP helper to prepared complexes. Explicit metadata avoids parsing group labels from filenames. Raw PLIP-dependent extraction needs separate validation in the structural environment.

## Configuration

Use `config.local.json` as described in [the workflow guide](../docs/WORKFLOW.md). Inputs must be authorized for local analysis. Outputs are cleared in version control; no research result is asserted by this notebook.

In [ ]:
from pathlib import Path
import sys
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'analysis/interactions.py').is_file()), None)
if root is None:
    raise FileNotFoundError('Start Jupyter from this repository')
sys.path.insert(0, str(root))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from analysis.notebook_setup import load_config, load_manifest
config = load_config()
out = config['output_dir']


## Extract events and preserve zero-contact observations

In [ ]:
sys.path.insert(0, str(root / 'notebooks'))
from src.generate_graph_new import protein_ligand_interaction
manifest = load_manifest(config)
frames = []
for row in manifest.itertuples(index=False):
    events = protein_ligand_interaction(row.complex_path, ligand_id=[config['ligand_id']])
    if not events.empty:
        if not {'feature', 'prot_chain', 'prot_res_num'} <= set(events):
            raise ValueError('Unexpected interaction schema')
        frames.append(events.assign(compound_id=row.compound_id, pose_id=row.pose_id, group=row.group))
columns = ['feature', 'prot_chain', 'prot_res_num', 'compound_id', 'pose_id', 'group']
events = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=columns)
events.to_csv(out / 'interaction_events.csv', index=False)
manifest[['compound_id', 'pose_id', 'group']].to_csv(out / 'pose_metadata.csv', index=False)
events.head()